In [1]:
import hail as hl
# Initialize Hail 
hl.init(default_reference = 'GRCh38')

Loading BokehJS ...

/opt/conda/miniconda3/lib/python3.10/site-packages/hail/context.py:347: UserWarning:

Using hl.init with a default_reference argument is deprecated. To set a default reference genome after initializing hail, call `hl.default_reference` with an argument to set the default reference genome.

/opt/conda/miniconda3/lib/python3.10/site-packages/hailtop/aiocloud/aiogoogle/user_config.py:44: UserWarning:

Reading spark-defaults.conf to determine GCS requester pays configuration. This is deprecated. Please use `hailctl config set gcs_requester_pays/project` and `hailctl config set gcs_requester_pays/buckets`.

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


SPARKMONITOR_LISTENER: Started SparkListener for Jupyter Notebook
SPARKMONITOR_LISTENER: Port obtained from environment: 45197
SPARKMONITOR_LISTENER: Application Started: application_1710774073885_0001 ...Start Time: 1710775045434


Running on Apache Spark version 3.3.2
SparkUI available at http://ibd-exome-m.c.daly-ibd.internal:42543
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.128-eead8100a1c1
LOGGING: writing to /home/hail/hail-20240318-1517-0.2.128-eead8100a1c1.log


In [4]:
main_mt = hl.read_matrix_table("gs://ibd-exomes-gnomad-subset/QC_round3/4.join_tables/twist_nextera.mt")
main_mt = main_mt.select_entries('GT', 'AD', 'DP', 'GQ', 'PL')
main_mt = main_mt.drop("sample_qc", "variant_qc", "capture")
# main_mt.describe()
main_mt_variants = main_mt.rows()


In [5]:
moayeddi_mt = hl.read_matrix_table("gs://ibd-exomes-gnomad-subset/QC_round2/4.moayeddi_merge/moayyedi_QCed.mt")
moayeddi_mt = moayeddi_mt.select_entries('GT', 'AD', 'DP', 'GQ', 'PL')
moayeddi_mt = moayeddi_mt.filter_rows(hl.is_defined(main_mt_variants[moayeddi_mt.locus, moayeddi_mt.alleles]))
moayeddi_mt.count()

(789058, 1246)

In [6]:
merged_mt = main_mt.union_cols(moayeddi_mt, row_join_type='outer')
merged_mt.write("gs://ibd-exomes-gnomad-subset/QC_round3/5.merge_moayeddi/merged.mt", overwrite = True)
merged_mt.count()

2024-03-18 15:21:41.020 Hail: INFO: wrote table with 789058 rows in 220 partitions to /tmp/__iruid_849-hZ68cajAjbldQiScWYieYE
IOPub message rate exceeded.=======================>      (42135 + 120) / 48180]
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)



(22285775, 173330)